# Two-Day PyTorch Interview Sprint

**AI Engineer / Machine Learning Engineer core practice · 10–12 focused hours**

This workbook targets the smallest set of PyTorch skills with the highest interview value: shapes, tabular MLPs, loss/target matching, training and validation loops, data loading, devices, and debugging.

| Day | Block | Topic | Time |
|---|---:|---|---:|
| 1 | 1 | Pre-assessment and tensor shape review | 45 min |
| 1 | 2 | Autograd and `nn.Module` | 60 min |
| 1 | 3 | Loss selection and target formats | 60 min |
| 1 | 4 | Training and validation loops | 2 hr |
| 1 | 5 | Closed-book rewrite and review | 30 min |
| 2 | 1 | Dataset and DataLoader | 60 min |
| 2 | 2 | End-to-end tabular multiclass project | 2 hr |
| 2 | 3 | Debugging drills | 75 min |
| 2 | 4 | Mock coding interview | 60 min |
| 2 | 5 | Rapid-fire Q&A and final recall | 45 min |

### If I Fall Behind

Preserve these five items, in order: **training loop → loss/output/target matching → shape reasoning → Dataset/DataLoader → one end-to-end multiclass classifier**. Skip quick reviews and extra verbal questions first.


## 0. How to Use This Notebook

Use this loop repeatedly:

> **Predict → write from memory → run → validate with assert → inspect the mistake → explain it aloud → repeat without notes**

1. Attempt each prompt before revealing or running its solution.
2. Type important patterns; do not copy/paste them.
3. Say every shape and dtype assumption aloud.
4. Keep a short mistake log.
5. Rewrite the training loop at least three times.
6. Stop low-priority work when its time box ends.

> **Do not merely read working code. Predict it, type it, run it, and explain it.**

| Mistake | Why it happened | Correct rule | Can I now write it? |
|---|---|---|---|
| Example: float targets for CE | confused binary and multiclass | CE targets are class indices with `torch.long` | ☐ |
|  |  |  | ☐ |
|  |  |  | ☐ |


In [ ]:
import copy
import random
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")


## 1. Pre-Assessment

**⏱ Suggested time: 20–25 minutes · closed book**

Do not run code yet. Record one answer per question, then score yourself against the compact key.

1. `(8, 12)` reshaped to `(4, -1)` becomes what shape?
2. What is the shape of `x.unsqueeze(1)` for `x.shape == (32, 10)`?
3. What can go wrong with bare `squeeze()` when the batch size is 1?
4. What shape results from transposing dimensions 0 and 1 of `(32, 10)`?
5. For `x.shape == (2, 3, 4)`, what is `x.permute(0, 2, 1).shape`?
6. Explain the shape difference between `cat([a, b], dim=0)` and `stack([a, b], dim=0)` when both inputs are `(8, 12)`.
7. A tabular classifier receives 10 features and predicts 4 classes. What are its input and output batch shapes?
8. Choose the loss and target dtype for one continuous value per row.
9. Choose the loss, output shape, and target dtype for binary classification.
10. Choose the loss, output shape, target shape, and dtype for 4-class classification.
11. Order: `loss.backward()`, forward, `optimizer.step()`, `optimizer.zero_grad()`, compute loss.
12. What changes when calling `model.train()` versus `model.eval()`?
13. Why is `model.eval()` different from `torch.inference_mode()`?
14. Define `Dataset` versus `DataLoader` in one sentence each.
15. A model is on an accelerator and `X` is on CPU. What is wrong, and why can a gradient double across steps?

<details><summary><strong>Compact answers and routing</strong></summary>

1. `(4, 24)`.
2. `(32, 1, 10)`.
3. It can remove the batch dimension; prefer `squeeze(dim)`.
4. `(10, 32)`.
5. `(2, 4, 3)`.
6. `cat`: `(16, 12)`; `stack`: `(2, 8, 12)`.
7. Input `(B, 10)`, output logits `(B, 4)`.
8. MSE and floating targets.
9. BCE-with-logits, `(B, 1)` or consistent `(B,)`, floating targets.
10. Cross entropy, `(B, 4)` logits, `(B,)` long class indices.
11. zero → forward → loss → backward → step.
12. It toggles training/evaluation behavior in affected layers.
13. `eval()` changes layer behavior; inference mode disables gradient recording.
14. Dataset retrieves one sample; DataLoader batches/shuffles/iterates samples.
15. Devices are incompatible; gradients accumulate unless cleared.

**Score /15:** 0–6 do every section; 7–10 do every Must Know exercise and skim reviews; 11–13 focus on closed-book coding and debugging; 14–15 go to the project and mock interview, then revisit mistakes.
</details>


## 2. Tensor Shapes and Operations

**🔥 Must know · ⏱ 45 minutes**

Say the shape before running anything. For tabular input, `B` is batch size, `F` features, `H` hidden width, and `C` classes.

| Operation | Example | Shape effect | Interview warning |
|---|---|---|---|
| `reshape` | `x.reshape(B, -1)` | rearranges elements into a compatible shape | may copy; element count must match |
| `view` | `x.view(B, -1)` | view with compatible strides | can fail after transpose; use `reshape` or `.contiguous().view(...)` |
| `flatten` | `x.flatten(1)` | combines dimensions from `start_dim` | preserve batch dimension deliberately |
| `unsqueeze` | `x.unsqueeze(1)` | inserts size-1 dimension | dimension index matters |
| `squeeze` | `x.squeeze(1)` | removes a specified size-1 dimension | bare `squeeze()` may remove batch dimension |
| `transpose` | `x.transpose(0, 1)` | swaps two dimensions | often makes a non-contiguous view |
| `permute` | `x.permute(0, 2, 1)` | reorders all dimensions | supply every dimension exactly once |
| `cat` | `torch.cat([a,b], dim=1)` | grows an existing dimension | all other dimensions must match |
| `stack` | `torch.stack([a,b], dim=0)` | inserts a new dimension | inputs must have identical shapes |
| `sum` | `x.sum(dim=1)` | removes reduced dimension | `keepdim=True` keeps it at size 1 |
| `mean` | `x.mean(dim=0)` | removes reduced dimension | integer tensors require conversion for mean |
| `argmax` | `logits.argmax(dim=1)` | removes class dimension | multiclass predictions use class dimension |
| matrix multiply | `(B,F) @ (F,H)` | returns `(B,H)` | inner dimensions must match |

### Shape drill: answer on paper first

1. `(8,12).reshape(4,-1)`
2. `(32,10).unsqueeze(0)`
3. `(32,10).unsqueeze(2)`
4. `(1,10).squeeze()`
5. `(1,10).squeeze(0)`
6. `(32,10).transpose(0,1)`
7. `(2,3,4).permute(2,0,1)`
8. `(32,10).sum(dim=1)`
9. `(32,10).sum(dim=1, keepdim=True)`
10. `(32,10) + (10,)`
11. `(B,F) @ (F,H)`
12. `cat` two `(32,10)` tensors at `dim=0`
13. `cat` them at `dim=1`
14. `stack` them at `dim=0`
15. `stack` them at `dim=2`
16. `(32,4).argmax(dim=1)`
17. Why is `argmax(dim=0)` wrong for per-row class predictions?
18. `(8,3,4).flatten(1)`
19. Boolean indexing `(8,12)` with a row mask `(8,)`
20. `torch.where(condition, a, b)` with three broadcastable `(32,10)` operands
21. Can `x.transpose(0,1).view(-1)` always work?
22. What is safer than bare `squeeze()` for logits shaped `(1,1)`?

<details><summary><strong>Shape answers</strong></summary>

1 `(4,24)`; 2 `(1,32,10)`; 3 `(32,10,1)`; 4 `(10,)` (the trap); 5 `(10,)`; 6 `(10,32)`; 7 `(4,2,3)`; 8 `(32,)`; 9 `(32,1)`; 10 `(32,10)` by broadcasting; 11 `(B,H)`; 12 `(64,10)`; 13 `(32,20)`; 14 `(2,32,10)`; 15 `(32,10,2)`; 16 `(32,)`; 17 it compares different examples for each class; 18 `(8,12)`; 19 `(number_of_true_rows,12)`; 20 `(32,10)`; 21 no, its strides may be incompatible; 22 `squeeze(1)` or keep a consistent `(B,1)` contract.
</details>


### ✍️ Write from memory: tensor operations

Before running the solution, write code that creates a `(8,12)` tensor, selects its first two rows and last three columns, reshapes it to `(4,24)`, computes row means with and without `keepdim`, and combines two copies using both `cat` and `stack`.

```python
# Blank version: fill each operation in a scratch cell.
x = ...
selection = ...
reshaped = ...
row_means = ...
row_means_kept = ...
concatenated = ...
stacked = ...
```

Most likely mistake: treating `stack` as concatenation. `stack` adds a dimension; `cat` grows an existing one.


In [ ]:
# Runnable solution and validation
x = torch.arange(8 * 12, dtype=torch.float32).reshape(8, 12)
selection = x[:2, -3:]
reshaped = x.reshape(4, 24)
row_means = x.mean(dim=1)
row_means_kept = x.mean(dim=1, keepdim=True)
concatenated = torch.cat([x, x], dim=1)
stacked = torch.stack([x, x], dim=0)

assert x.shape == (8, 12) and x.ndim == 2
assert x.dtype == torch.float32 and x.device.type == "cpu"
assert selection.shape == (2, 3) and selection[0, 0].item() == 9
assert reshaped.shape == (4, 24)
assert row_means.shape == (8,) and row_means_kept.shape == (8, 1)
assert concatenated.shape == (8, 24) and stacked.shape == (2, 8, 12)


### 🛠 Debug this: transpose, masks, broadcasting, and predictions

Predict every result. A common wrong answer is to assume `view` always works or to use `argmax(dim=0)` for `(B,C)` logits.

```python
# Broken idea
flat = features.transpose(0, 1).view(-1)
predictions = logits.argmax(dim=0)
```


In [ ]:
features = torch.arange(24, dtype=torch.float32).reshape(4, 6)
transposed = features.transpose(0, 1)
flat = transposed.reshape(-1)  # or transposed.contiguous().view(-1)
row_mask = features[:, 0] >= 12
selected_rows = features[row_mask]
replaced = torch.where(features > 10, features, torch.zeros_like(features))
weights = torch.ones(6, 3)
hidden = features @ weights
logits = torch.tensor([[0.1, 0.8, -0.1], [2.0, 1.0, 0.0]])
predictions = logits.argmax(dim=1)

assert not transposed.is_contiguous()
assert flat.shape == (24,) and selected_rows.shape == (2, 6)
assert replaced.shape == features.shape and hidden.shape == (4, 3)
assert predictions.shape == (2,) and predictions.tolist() == [1, 0]


### Five-minute closed-book shape quiz

Write only the answers, then run the validation below.

1. `(32,10) @ (10,4)` → ?
2. Mean over the feature dimension with `keepdim=True` → ?
3. Stack three `(8,12)` tensors at dimension 1 → ?
4. Select predictions from `(32,4)` logits → expression and shape?
5. Safely remove only the last singleton dimension of `(1,10,1)` → expression and shape?


In [ ]:
# Runnable answer key
quiz_x = torch.randn(32, 10)
quiz_logits = quiz_x @ torch.randn(10, 4)
quiz_mean = quiz_x.mean(dim=1, keepdim=True)
quiz_stack = torch.stack([torch.zeros(8, 12)] * 3, dim=1)
quiz_predictions = quiz_logits.argmax(dim=1)
quiz_squeezed = torch.zeros(1, 10, 1).squeeze(-1)
assert quiz_logits.shape == (32, 4)
assert quiz_mean.shape == (32, 1)
assert quiz_stack.shape == (8, 3, 12)
assert quiz_predictions.shape == (32,)
assert quiz_squeezed.shape == (1, 10)


## 3. Autograd Essentials

**🔥 Must know · ⏱ 35 minutes**

PyTorch records tensor operations in a computation graph when gradients are enabled. `backward()` traverses that graph and stores gradients on leaf tensors such as model parameters. Gradients accumulate by default, so a normal independent step clears old gradients. Validation should avoid building an unnecessary graph.

`detach()` returns a tensor disconnected from the current graph; moving a tensor to CPU changes its device and does not inherently disconnect it.

### Predict, then prove

```python
# What is x.grad after backward? What happens if gradients are added again?
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x
y.backward()
```

Likely mistake: expecting `.grad` before `backward()`, or expecting a later backward call to replace the old gradient.


In [ ]:
x_grad_demo = torch.tensor(3.0, requires_grad=True)
y_grad_demo = x_grad_demo ** 2 + 2 * x_grad_demo
assert x_grad_demo.grad is None
y_grad_demo.backward()
assert torch.isclose(x_grad_demo.grad, torch.tensor(8.0))

# Two fresh graphs, same leaf: gradients add unless cleared.
z_grad_demo = torch.tensor(2.0, requires_grad=True)
(z_grad_demo ** 2).backward()
first_grad = z_grad_demo.grad.item()
(z_grad_demo ** 2).backward()
assert first_grad == 4.0 and z_grad_demo.grad.item() == 8.0
z_grad_demo.grad = None
(z_grad_demo ** 2).backward()
assert z_grad_demo.grad.item() == 4.0


### ✍️ One model gradient experiment

Before running, predict which tensors receive gradients. Then write a forward pass, scalar loss, `backward()`, `zero_grad()`, an inference-only pass, and a detached copy.

```python
model = nn.Linear(3, 1)
# Write the experiment in a scratch cell.
```

Common mistake: `model.eval()` alone does **not** disable Autograd.


In [ ]:
gradient_model = nn.Linear(3, 1)
gradient_optimizer = torch.optim.SGD(gradient_model.parameters(), lr=0.1)
gradient_features = torch.ones(4, 3)
gradient_targets = torch.zeros(4, 1)

gradient_optimizer.zero_grad()
gradient_outputs = gradient_model(gradient_features)
gradient_loss = nn.MSELoss()(gradient_outputs, gradient_targets)
gradient_loss.backward()
assert all(parameter.grad is not None for parameter in gradient_model.parameters())

gradient_optimizer.zero_grad(set_to_none=True)
assert all(parameter.grad is None for parameter in gradient_model.parameters())
with torch.inference_mode():
    inference_outputs = gradient_model(gradient_features)
assert not inference_outputs.requires_grad
assert not gradient_outputs.detach().requires_grad
assert gradient_outputs.cpu().requires_grad  # device move alone does not detach


### 🧠 Say this aloud

- Why did a gradient double? Because a second backward contribution was added to the existing `.grad`.
- Why is `param.grad` `None` before backward? No gradient has been computed and stored yet.
- Why disable gradient tracking in validation? It saves graph-building work and prevents accidental gradient use.
- `detach()` versus CPU? Detach breaks graph tracking; `.cpu()` changes placement.
- Does `model.eval()` disable Autograd? No.

**One-minute model answer:** “During a forward pass, PyTorch records differentiable operations in a dynamic computation graph. Calling `loss.backward()` applies the chain rule and stores gradients on leaf parameters. Those gradients accumulate, so I clear them before each independent training step. During validation I use evaluation mode for layer behavior and inference mode to avoid recording a graph.”


## 4. Build an MLP with `nn.Module`

**🔥 Must know · ⏱ 35 minutes**

An `nn.Module` registers parameter-bearing submodules, exposes `parameters()`, and integrates hooks and call behavior. `super().__init__()` initializes that machinery. `forward()` describes computation, but call `model(x)` so the module call path can do its full bookkeeping.

For `(B,10)` input and 3 classes, the last layer has 3 outputs and returns raw `(B,3)` logits. Do not add a probability conversion before cross entropy.

### ✍️ Write from memory

```python
class TabularMLP(nn.Module):
    def __init__(self, num_features, hidden_size, num_classes):
        # fill initialization, 10 -> hidden -> 3
        ...
    def forward(self, x):
        ...
```

Predict the intermediate shapes for `B=8`, hidden size 16. Manually count parameters: `(10×16 + 16) + (16×3 + 3)`.


In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, num_features: int, hidden_size: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        return self.net(x)


batch_size, num_features, hidden_size, num_classes = 8, 10, 16, 3
mlp = TabularMLP(num_features, hidden_size, num_classes)
mlp_features = torch.randn(batch_size, num_features)
mlp_logits = mlp(mlp_features)
manual_parameter_count = (10 * 16 + 16) + (16 * 3 + 3)
actual_parameter_count = sum(parameter.numel() for parameter in mlp.parameters() if parameter.requires_grad)

assert mlp.net[0](mlp_features).shape == (batch_size, hidden_size)
assert mlp_logits.shape == (batch_size, num_classes)
assert manual_parameter_count == actual_parameter_count == 227
assert {name for name, _ in mlp.named_parameters()} == {
    "net.0.weight", "net.0.bias", "net.2.weight", "net.2.bias"
}


### 🛠 Registration and initialization bugs

```python
# Broken A: layers in an ordinary Python list are not registered.
self.layers = [nn.Linear(10, 16), nn.Linear(16, 3)]

# Broken B: Module initialization is missing.
class Broken(nn.Module):
    def __init__(self):
        self.layer = nn.Linear(10, 3)
```

Fix A with `nn.ModuleList` or, here, `nn.Sequential`. Fix B with `super().__init__()` before assigning submodules. Most likely mistake: creating parameterized layers locally inside `forward`, which creates fresh unregistered parameters on every call.


In [ ]:
class RegisteredMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(10, 16), nn.GELU(), nn.Linear(16, 3))

    def forward(self, features):
        return self.layers(features)


registered_model = RegisteredMLP()
assert len(list(registered_model.parameters())) == 4
assert registered_model(torch.randn(5, 10)).shape == (5, 3)


## 5. Match Task, Output, Target, and Loss

**🔥 Highest-value reference · ⏱ 60 minutes**

| Task | Model output | Target | Target dtype | Loss | Prediction |
|---|---|---|---|---|---|
| Regression | `(B,1)` or `(B,)` | continuous value | floating point | `nn.MSELoss()` | raw output |
| Binary classification | `(B,1)` or consistently `(B,)` raw logits | 0/1 | floating point | `nn.BCEWithLogitsLoss()` | sigmoid then threshold |
| Multiclass classification | `(B,C)` raw logits | class index `(B,)` | `torch.long` | `nn.CrossEntropyLoss()` | `argmax(dim=1)` |
| Multilabel classification | `(B,C)` raw logits | independent 0/1 labels `(B,C)` | floating point | `nn.BCEWithLogitsLoss()` | sigmoid then per-class threshold |

> **CrossEntropyLoss: logits `(B,C)`; targets `(B,)`; targets `torch.long`.**

Logits are raw scores, not probabilities. Cross entropy combines a log-softmax-like calculation with negative log likelihood; BCE-with-logits likewise combines sigmoid behavior with its loss for numerical stability. The prediction task determines the final layer size and target contract.

### Fifteen scenario drills

For each, state: **output dimension · target shape · dtype · loss · prediction conversion**.

1. House price (one value). 2. Monthly demand (one value). 3. Customer churn yes/no. 4. Fraud yes/no. 5. Employee attrition yes/no. 6. Credit risk: low/medium/high. 7. Product category: 8 exclusive categories. 8. Machine status: 3 exclusive states. 9. Customer segment: 5 exclusive groups. 10. Support ticket priority: 4 exclusive levels. 11. Independent customer interests across 6 tags. 12. Multiple applicable compliance flags across 4 flags. 13. Three-class sensor diagnosis. 14. Binary loan default. 15. Continuous time-to-completion estimate.

<details><summary><strong>Compact answer pattern</strong></summary>

1,2,15: 1 · `(B,1)` · float · MSE · raw. 3,4,5,14: 1 · `(B,1)` · float · BCE-with-logits · sigmoid/threshold. 6,7,8,9,10,13: `C` · `(B,)` · long · cross entropy · `argmax(1)`. 11,12: `C` · `(B,C)` · float · BCE-with-logits · sigmoid/per-class threshold.
</details>


### ✍️ Implement the four loss contracts

Write each loss call on paper first. Avoid these common errors: float class indices for CE, integer BCE targets, probability conversion before a logits-based loss, bare `squeeze()`, and the wrong final width.

```python
# Fill four scalar losses and predictions for B=5.
regression_loss = ...
binary_loss = ...
multiclass_loss = ...
multilabel_loss = ...
```


In [ ]:
loss_batch_size = 5
regression_outputs = torch.randn(loss_batch_size, 1)
regression_targets = torch.randn(loss_batch_size, 1)
binary_logits = torch.randn(loss_batch_size, 1)
binary_targets = torch.randint(0, 2, (loss_batch_size, 1)).float()
multiclass_logits = torch.randn(loss_batch_size, 4)
multiclass_targets = torch.randint(0, 4, (loss_batch_size,), dtype=torch.long)
multilabel_logits = torch.randn(loss_batch_size, 6)
multilabel_targets = torch.randint(0, 2, (loss_batch_size, 6)).float()

regression_loss = nn.MSELoss()(regression_outputs, regression_targets)
binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_targets)
multiclass_loss = nn.CrossEntropyLoss()(multiclass_logits, multiclass_targets)
multilabel_loss = nn.BCEWithLogitsLoss()(multilabel_logits, multilabel_targets)
binary_predictions = (torch.sigmoid(binary_logits) >= 0.5).long()
multiclass_predictions = multiclass_logits.argmax(dim=1)

for current_loss in [regression_loss, binary_loss, multiclass_loss, multilabel_loss]:
    assert current_loss.ndim == 0 and torch.isfinite(current_loss)
assert binary_predictions.shape == (loss_batch_size, 1)
assert multiclass_predictions.shape == (loss_batch_size,)


### 🛠 Five loss-contract repairs

```python
# A: nn.CrossEntropyLoss()(logits, class_indices.float())
# B: nn.BCEWithLogitsLoss()(binary_logits, class_indices)
# C: nn.CrossEntropyLoss()(torch.softmax(logits, 1), class_indices)
# D: nn.BCEWithLogitsLoss()(binary_logits.squeeze(), targets.squeeze())
# E: nn.Linear(hidden, 3) for a 4-class task
```

Diagnose before running the fixes. A needs long indices; B needs aligned floating binary targets; C needs raw logits; D should preserve the batch contract with `squeeze(1)` if needed; E needs 4 outputs. The likely incorrect instinct is to “fix” a shape error by indiscriminately squeezing.


In [ ]:
repair_features = torch.randn(7, 10)
repair_hidden = nn.ReLU()(nn.Linear(10, 8)(repair_features))
repair_logits = nn.Linear(8, 4)(repair_hidden)
repair_targets = torch.randint(0, 4, (7,), dtype=torch.long)
repair_ce_loss = nn.CrossEntropyLoss()(repair_logits, repair_targets)

repair_binary_logits = torch.randn(7, 1)
repair_binary_targets = torch.randint(0, 2, (7, 1)).float()
repair_binary_loss = nn.BCEWithLogitsLoss()(repair_binary_logits, repair_binary_targets)

assert repair_logits.shape == (7, 4)
assert repair_targets.shape == (7,) and repair_targets.dtype == torch.long
assert repair_ce_loss.ndim == 0 and torch.isfinite(repair_ce_loss)
assert repair_binary_loss.ndim == 0 and torch.isfinite(repair_binary_loss)


## 6. Training Loop from Memory

**🔥 Deepest focus · ⏱ 2 hours**

Memorize the order:

> `model.train()` → batch → move batch → `zero_grad()` → forward → loss → `backward()` → `step()` → detached scalar metrics

Learning rate controls update size. `zero_grad()` clears accumulated gradients; `step()` updates parameters. AdamW is a strong common default; SGD is simpler and remains conceptually important.


### Stage 1: one optimization step

Write the missing lines in a scratch cell. Predict when gradients appear and when parameters change.

```python
model.train()
optimizer.________
logits = ________
loss = ________
________
________
```

Most likely mistake: calling `step()` before `backward()` or clearing gradients between them.


In [ ]:
torch.manual_seed(SEED)
step_model = TabularMLP(10, 16, 3).to(device)
step_optimizer = torch.optim.AdamW(step_model.parameters(), lr=1e-3)
step_loss_fn = nn.CrossEntropyLoss()
step_features = torch.randn(12, 10, device=device)
step_targets = torch.randint(0, 3, (12,), dtype=torch.long, device=device)
before_step = {name: parameter.detach().clone() for name, parameter in step_model.named_parameters()}

step_model.train()
step_optimizer.zero_grad()
step_logits = step_model(step_features)
step_loss = step_loss_fn(step_logits, step_targets)
assert step_loss.ndim == 0 and torch.isfinite(step_loss)
step_loss.backward()
assert all(parameter.grad is not None for parameter in step_model.parameters())
step_optimizer.step()
assert any(not torch.equal(before_step[name], parameter.detach())
           for name, parameter in step_model.named_parameters())


### Stage 2: `train_one_epoch`

First write it closed-book. Correct aggregation weights each batch mean by its number of examples, so an incomplete final batch is handled correctly. A plain mean of batch means gives a small final batch the same weight as a full one.

```python
def train_one_epoch(model, data_loader, loss_fn, optimizer, device):
    # mode, counters, batch loop, device, zero, forward, loss,
    # backward, step, weighted metrics, return loss and accuracy
```


In [ ]:
def train_one_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for features, targets in data_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(features)
        loss = loss_fn(logits, targets)
        assert loss.ndim == 0 and torch.isfinite(loss)
        loss.backward()
        optimizer.step()

        batch_size_now = features.size(0)
        total_loss += loss.item() * batch_size_now
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_examples += batch_size_now

    epoch_loss = total_loss / total_examples
    epoch_accuracy = total_correct / total_examples
    assert epoch_loss > 0.0 and torch.isfinite(torch.tensor(epoch_loss))
    assert 0.0 <= epoch_accuracy <= 1.0
    return epoch_loss, epoch_accuracy


### Stage 3: train on learnable data, then rewrite twice

Before running, predict the batch shapes and why loss should fall. After running:

1. **✍️ Rewrite #1:** reproduce `train_one_epoch` with only the order mnemonic.
2. **✍️ Rewrite #2:** close all notes and reproduce it again, including weighted aggregation.
3. Explain each line aloud.

Blank version:

```python
model.____()
for features, targets in loader:
    # device → zero → forward → loss → backward → step → metrics
```


In [ ]:
torch.manual_seed(SEED)
train_features_core = torch.randn(384, 10)
teacher_weights_core = torch.randn(10, 3)
train_targets_core = (train_features_core @ teacher_weights_core).argmax(dim=1)
train_loader_core = DataLoader(
    TensorDataset(train_features_core, train_targets_core),
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
train_model_core = TabularMLP(10, 24, 3).to(device)
train_optimizer_core = torch.optim.AdamW(train_model_core.parameters(), lr=1e-2)
train_loss_fn_core = nn.CrossEntropyLoss()

core_history = []
for epoch in range(10):
    core_history.append(train_one_epoch(
        train_model_core, train_loader_core, train_loss_fn_core,
        train_optimizer_core, device
    ))

first_core_loss = core_history[0][0]
final_core_loss, final_core_accuracy = core_history[-1]
print(f"Training loss: {first_core_loss:.3f} -> {final_core_loss:.3f}; accuracy: {final_core_accuracy:.3f}")
assert final_core_loss < first_core_loss
assert 0.0 <= final_core_accuracy <= 1.0


### Ten broken training loops: diagnose symptom and smallest fix

| # | Broken idea | Likely symptom | Smallest fix / prevention rule |
|---:|---|---|---|
| 1 | omit `zero_grad()` | gradients mix across batches | clear before each independent step |
| 2 | omit `backward()` | parameters do not learn | compute gradients before `step()` |
| 3 | omit `step()` | gradients exist but weights do not change | call `step()` after backward |
| 4 | `step()` before backward | stale/no gradient update | use zero → forward → loss → backward → step |
| 5 | zero between backward and step | update has no gradients | never clear in that interval |
| 6 | loss from `logits.detach()` | backward error/no model path | compute training loss from attached logits |
| 7 | append `loss` tensors | retained graphs/memory growth | log `loss.item()` |
| 8 | model and batch on different devices | device mismatch error | move model and both batch tensors |
| 9 | CE target is float | dtype error | class indices must be `long` |
| 10 | model remains in evaluation mode | affected training layers behave incorrectly | call `model.train()` once per training epoch |

For each row, cover: what is wrong, its symptom, and the smallest fix. The runnable reference loop below validates the repaired order.


In [ ]:
# Runnable repair shared by all ten drills
repair_train_model = TabularMLP(10, 12, 3).to(device)
repair_train_optimizer = torch.optim.AdamW(repair_train_model.parameters(), lr=1e-3)
repair_train_features = torch.randn(9, 10).to(device)
repair_train_targets = torch.randint(0, 3, (9,), dtype=torch.long).to(device)
repair_train_model.train()
repair_train_optimizer.zero_grad()
repair_train_logits = repair_train_model(repair_train_features)  # attached, same device
repair_train_loss = nn.CrossEntropyLoss()(repair_train_logits, repair_train_targets)
repair_train_loss.backward()
repair_grad_norm = sum(p.grad.norm().item() for p in repair_train_model.parameters())
repair_train_optimizer.step()
repair_logged_loss = repair_train_loss.item()
assert repair_grad_norm > 0 and isinstance(repair_logged_loss, float)


## 7. Validation Loop and Metrics

**🔥 Must know · ⏱ 45 minutes**

`model.eval()` changes the behavior of layers such as Dropout; `torch.inference_mode()` disables gradient tracking and reduces overhead. Validation uses both, has no backward pass or optimizer step, aggregates over all examples, and must not update parameters.

### ✍️ Validation rewrite #1

```python
def evaluate(model, data_loader, loss_fn, device):
    # eval → inference context → batches → device → forward → loss/predictions
    # → weighted totals → return average loss and accuracy
```

Likely mistakes: `argmax(dim=0)`, averaging batch accuracies equally, or assuming `eval()` disables gradients.


In [ ]:
def evaluate(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.inference_mode():
        for features, targets in data_loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            loss = loss_fn(logits, targets)
            predictions = logits.argmax(dim=1)

            batch_size_now = features.size(0)
            total_loss += loss.item() * batch_size_now
            total_correct += (predictions == targets).sum().item()
            total_examples += batch_size_now

    average_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    assert 0.0 <= accuracy <= 1.0
    return average_loss, accuracy


### Prove validation does not update parameters

Before running, explain why state should be identical afterward. Then **✍️ rewrite validation #2** from memory.


In [ ]:
validation_features = torch.randn(73, 10)
validation_targets = (validation_features @ teacher_weights_core).argmax(dim=1)
validation_loader = DataLoader(
    TensorDataset(validation_features, validation_targets),
    batch_size=16, shuffle=False, num_workers=0,
)
state_before_validation = copy.deepcopy(train_model_core.state_dict())

# Remove old gradients so a clean inference pass can prove it creates none.
train_optimizer_core.zero_grad(set_to_none=True)
validation_loss, validation_accuracy = evaluate(
    train_model_core, validation_loader, train_loss_fn_core, device
)
assert all(parameter.grad is None for parameter in train_model_core.parameters())
assert all(torch.equal(state_before_validation[name], value)
           for name, value in train_model_core.state_dict().items())

with torch.inference_mode():
    validation_batch_logits = train_model_core(validation_features[:5].to(device))
    validation_batch_predictions = validation_batch_logits.argmax(dim=1)
assert validation_batch_predictions.shape == (5,)
assert 0.0 <= validation_accuracy <= 1.0
print(f"Validation loss: {validation_loss:.3f}; accuracy: {validation_accuracy:.3f}")


### `train()` and `eval()` are observable

Dropout randomly zeros values in training mode and becomes an identity-like operation in evaluation mode. This is separate from gradient tracking.


In [ ]:
dropout_demo = nn.Dropout(p=0.75)
dropout_input = torch.ones(128)
dropout_demo.train()
torch.manual_seed(SEED)
dropout_training_output = dropout_demo(dropout_input)
dropout_demo.eval()
dropout_evaluation_output = dropout_demo(dropout_input)
assert not torch.equal(dropout_training_output, dropout_input)
assert torch.equal(dropout_evaluation_output, dropout_input)
assert torch.is_grad_enabled()  # eval mode did not globally disable Autograd


### 🛠 Validation debugging prompts

- `eval()` without inference mode: predictions may be correct, but a graph is unnecessarily recorded. Add the context.
- Inference mode without `eval()`: gradients are off, but affected layers still use training behavior. Call `eval()`.
- `optimizer.step()` in validation: parameters can change. Remove all update calls and compare state before/after.
- `argmax(dim=0)`: returns one winning example per class, not one class per example. Use `dim=1`.
- Mean of per-batch accuracies: a small final batch is overweighted. Sum correct predictions and divide by total examples.


## 8. Day 1 Closed-Book Checkpoint

**⏱ 30 minutes · no notes and no teaching before the answer key**

1. Write a 10-feature, three-class tabular MLP. **2 points**
2. State the cross-entropy output, target shape, and dtype. **2 points**
3. Write one complete batch training step. **2 points**
4. Write a full training epoch with weighted loss and accuracy. **3 points**
5. Write a validation function. **3 points**
6. Explain correct loss aggregation for an incomplete last batch. **1 point**
7. Explain `train()` versus `eval()`. **1 point**
8. Explain `eval()` versus inference mode. **1 point**
9. Explain why gradients accumulate. **1 point**
10. Predict: `(8,12).reshape(3,-1)`; `(32,10).mean(1)`; `cat` two `(32,4)` at dim 1; `stack` them at dim 0; `(32,4).argmax(1)`. **4 points**

<details><summary><strong>Reference answers and 20-point rubric</strong></summary>

Use `TabularMLP`, `train_one_epoch`, and `evaluate` above as the coding key. CE consumes `(B,3)` raw logits and `(B,)` long targets. Weighted aggregation is `sum(batch_mean * batch_size) / total_examples`. Mode changes affected layer behavior; inference mode changes graph recording. Gradients add to `.grad` by default.

Shapes: the first reshape is invalid because 96 is not divisible by 3; `(32,)`; `(32,8)`; `(2,32,4)`; `(32,)`.

**17–20:** proceed to Day 2. **13–16:** rewrite training and validation once, then proceed. **0–12:** repeat Sections 5–7 before starting the project.
</details>


## 9. Dataset and DataLoader

**🔥 Must know · ⏱ 60 minutes**

> **Dataset → defines dataset size and retrieves one sample. DataLoader → batches, shuffles, and iterates over samples.**

Default collation stacks compatible samples. Training is normally shuffled to vary order; validation normally is not because order adds no value. `num_workers=0` is the safest notebook default. `drop_last=True` discards an incomplete last batch; normally keep validation examples with `False`.

`TensorDataset` is enough when aligned tensors already fit in memory. A custom Dataset is useful when sample retrieval needs logic.

### ✍️ Implement from memory

```python
class TabularDataset(Dataset):
    def __init__(self, features, targets): ...
    def __len__(self): ...
    def __getitem__(self, index): ...
```

Likely mistake: returning a batch from `__getitem__`; it should return one sample here.


In [ ]:
class TabularDataset(Dataset):
    def __init__(self, features, targets):
        if len(features) != len(targets):
            raise ValueError("features and targets must have equal length")
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.targets[index]


dataset_features = torch.randn(70, 10)
dataset_targets = torch.randint(0, 3, (70,), dtype=torch.long)
custom_dataset = TabularDataset(dataset_features, dataset_targets)
sample_features, sample_target = custom_dataset[0]
assert len(custom_dataset) == 70
assert sample_features.shape == (10,) and sample_target.shape == ()
assert sample_features.dtype == torch.float32 and sample_target.dtype == torch.long


### Eight DataLoader exercises

Predict before running:

1. Validate the custom Dataset.
2. Compare one sample to one batch.
3. For 70 samples and batch size 16, predict loader length using ceiling division.
4. Predict the final batch size with `drop_last=False`.
5. Compare lengths and final behavior when `drop_last=True`.
6. Explain why validation uses `shuffle=False`.
7. Rewrite the custom Dataset from memory.
8. Wrap the same tensors in `TensorDataset`.

```python
# Fill these in on paper first.
loader_length = ...
last_batch_size = ...
```

Most likely mistake: floor-dividing loader length even when the smaller last batch is retained.


In [ ]:
loader_keep_last = DataLoader(
    custom_dataset, batch_size=16, shuffle=False, drop_last=False, num_workers=0
)
loader_drop_last = DataLoader(
    custom_dataset, batch_size=16, shuffle=False, drop_last=True, num_workers=0
)
batch_features, batch_targets = next(iter(loader_keep_last))
all_keep_batches = list(loader_keep_last)
tensor_dataset = TensorDataset(dataset_features, dataset_targets)

assert sample_features.shape == (10,) and batch_features.shape == (16, 10)
assert batch_targets.shape == (16,) and batch_targets.dtype == torch.long
assert len(loader_keep_last) == (70 + 16 - 1) // 16 == 5
assert all_keep_batches[-1][0].shape == (6, 10)
assert len(loader_drop_last) == 70 // 16 == 4
assert len(tensor_dataset) == len(custom_dataset)
assert torch.equal(tensor_dataset[3][0], custom_dataset[3][0])


### 🟡 Optional note: pinned memory

`pin_memory=True` can help host-to-accelerator transfer in some setups, but it is not needed for this small workbook. Treat it as a measured performance option, not a correctness requirement.


## 10. End-to-End Tabular Multiclass Project

**🔥 Best final practical test · ⏱ 2 hours**

Before each code cell, decide aloud:

- One sample input and target shape?
- Batch input and target shape?
- Target dtype?
- Final layer size?
- Loss and prediction rule?
- Metric?
- Which loader shuffles?
- Where do device transfers belong?

Workflow: **understand one sample → inspect → split → datasets → loaders → model → loss/optimizer → train → validate → inspect predictions → debug → explain**.


### Step 1 — Generate and split learnable data

We need 12 floating features, 4 exclusive class indices, and an 80/20 split. Labels come from a fixed rule plus modest noise, so the problem is learnable.

```python
# Decide these before running:
# features.shape = ?; targets.shape = ?; targets.dtype = ?
```


In [ ]:
torch.manual_seed(SEED)
project_num_examples = 1600
project_num_features = 12
project_num_classes = 4
project_features = torch.randn(project_num_examples, project_num_features)
project_teacher = torch.randn(project_num_features, project_num_classes)
project_bias = torch.tensor([-0.25, 0.10, 0.20, -0.05])
project_scores = project_features @ project_teacher + project_bias
project_scores += 0.12 * torch.randn_like(project_scores)
project_targets = project_scores.argmax(dim=1).long()

split_index = int(0.8 * project_num_examples)
project_train_features = project_features[:split_index]
project_val_features = project_features[split_index:]
project_train_targets = project_targets[:split_index]
project_val_targets = project_targets[split_index:]

assert project_features.shape == (1600, 12)
assert project_targets.shape == (1600,) and project_targets.dtype == torch.long
assert project_train_features.shape == (1280, 12)
assert project_val_features.shape == (320, 12)
assert project_targets.unique().numel() == project_num_classes


### Step 2 — Dataset and loaders

Decide why only the training loader shuffles and whether to keep the last validation batch. Write both loaders before running the solution.


In [ ]:
project_train_dataset = TabularDataset(project_train_features, project_train_targets)
project_val_dataset = TensorDataset(project_val_features, project_val_targets)
project_train_loader = DataLoader(
    project_train_dataset, batch_size=64, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
project_val_loader = DataLoader(
    project_val_dataset, batch_size=64, shuffle=False, num_workers=0,
)
project_sample_features, project_sample_target = project_train_dataset[0]
project_batch_features, project_batch_targets = next(iter(project_train_loader))
assert project_sample_features.shape == (12,) and project_sample_target.shape == ()
assert project_batch_features.shape == (64, 12)
assert project_batch_targets.shape == (64,) and project_batch_targets.dtype == torch.long


### Step 3 — Model, objective, and optimizer

Write a `12 → 32 → 16 → 4` MLP. The output is raw logits; targets are class indices. Choose the loss and prediction rule before running.


In [ ]:
class ProjectMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(12, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4),
        )

    def forward(self, features):
        return self.net(features)


torch.manual_seed(SEED)
project_model = ProjectMLP().to(device)
project_loss_fn = nn.CrossEntropyLoss()
project_optimizer = torch.optim.AdamW(project_model.parameters(), lr=8e-3)
with torch.inference_mode():
    initial_project_logits = project_model(project_batch_features.to(device))
assert initial_project_logits.shape == (64, 4)


### Step 4 — Train and validate

Write the epoch driver first. Device transfers stay inside the reusable batch loops. Do not expect perfectly monotonic loss; require clear learning across the run.


In [ ]:
project_history = []
for project_epoch in range(14):
    project_train_loss, project_train_accuracy = train_one_epoch(
        project_model, project_train_loader, project_loss_fn, project_optimizer, device
    )
    project_val_loss, project_val_accuracy = evaluate(
        project_model, project_val_loader, project_loss_fn, device
    )
    project_history.append({
        "train_loss": project_train_loss,
        "train_accuracy": project_train_accuracy,
        "val_loss": project_val_loss,
        "val_accuracy": project_val_accuracy,
    })

final_project = project_history[-1]
print(
    f"Final training loss: {final_project['train_loss']:.3f}\n"
    f"Final validation loss: {final_project['val_loss']:.3f}\n"
    f"Final validation accuracy: {final_project['val_accuracy']:.3f}"
)
assert final_project["train_loss"] < project_history[0]["train_loss"]
assert final_project["val_accuracy"] > 0.65  # random chance is 0.25


### Step 5 — Inspect five predictions

Predict the tensor shapes before running. Outputs can be moved to CPU for ordinary inspection.


In [ ]:
project_model.eval()
with torch.inference_mode():
    five_project_logits = project_model(project_val_features[:5].to(device))
    five_project_predictions = five_project_logits.argmax(dim=1).cpu()
five_project_targets = project_val_targets[:5]
project_pairs = list(zip(five_project_targets.tolist(), five_project_predictions.tolist()))
print("Five (target, prediction) pairs:", project_pairs)
assert five_project_logits.shape == (5, 4)
assert five_project_predictions.shape == (5,)


### Project ablation/debug questions

1. Three outputs for four classes: class 3 can never be predicted and targets may be out of bounds.
2. Float targets: cross entropy rejects the dtype; keep long class indices.
3. Softmax before loss: redundant and numerically/gradient-wise inferior; pass raw logits.
4. Remove `zero_grad()`: gradients mix across batches.
5. Remove every hidden ReLU: the composition becomes effectively linear and loses nonlinear capacity.
6. Disable training shuffle: code remains correct, though batch order may make optimization less robust.
7. Update during validation: evaluation leaks into model state; remove backward/step and compare parameters.
8. CPU inputs with an accelerator model: device mismatch; move every batch tensor.

### 🧠 Two-minute interview walkthrough

**Template:** Problem · Data representation · Model/output · Loss/target · Training/validation · Metric · Bug checked · Next improvement.

**Example:** “I framed this as four-class classification over 12 floating features. Each batch is `(B,12)` and the MLP produces raw `(B,4)` logits; targets are `(B,)` long class indices, so I used cross entropy. I shuffled only training data, moved each batch to the model device, cleared gradients before backward, and aggregated loss and accuracy by example. Validation used both evaluation mode and inference mode, and accuracy finished clearly above the 25% baseline. I explicitly checked target dtype and parameter updates; next I would tune width or regularization using validation results.”


## 11. Debugging Drills

**🔥 Must know · ⏱ 75 minutes**

Use the same order every time:

> **1 shape → 2 dtype → 3 device → 4 `requires_grad`/gradients → 5 mode → 6 loss compatibility → 7 parameter change → 8 one batch → 9 overfit a tiny subset**

The helper below keeps inspection factual.


In [ ]:
def tensor_diagnostics(name, tensor):
    information = {
        "shape": tuple(tensor.shape),
        "dtype": str(tensor.dtype),
        "device": str(tensor.device),
        "requires_grad": tensor.requires_grad,
        "finite": bool(torch.isfinite(tensor).all().item()),
    }
    print(name, information)
    return information


diagnostic_result = tensor_diagnostics("project batch", project_batch_features)
assert diagnostic_result["shape"] == (64, 12)
assert diagnostic_result["finite"]


### 🛠 Debug drill 1: Shape 1 — incompatible matrix multiplication

**Broken snippet**

```python
`nn.Linear(12, 8)(torch.randn(5, 10))`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** A matrix multiplication shape error appears because 10 input features do not match 12 expected features.

**Prevention rule:** Inspect the last input dimension and the layer's `in_features` before training.


In [ ]:
# Corrected runnable solution
fixed_layer = nn.Linear(12, 8)
fixed_features = torch.randn(5, 12)
fixed_output = fixed_layer(fixed_features)
assert fixed_output.shape == (5, 8)


### 🛠 Debug drill 2: Shape 2 — wrong final output width

**Broken snippet**

```python
`logits = nn.Linear(16, 3)(hidden)  # task has four classes`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** The model cannot represent all classes; CE can also report an out-of-bounds target.

**Prevention rule:** Set the final width from the number of prediction classes.


In [ ]:
# Corrected runnable solution
fixed_hidden = torch.randn(6, 16)
fixed_logits = nn.Linear(16, 4)(fixed_hidden)
assert fixed_logits.shape == (6, 4)


### 🛠 Debug drill 3: Shape 3 — CE target has an extra dimension

**Broken snippet**

```python
`loss_fn(logits, targets.reshape(B, 1))`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Cross entropy expects one-dimensional class indices for ordinary multiclass batches.

**Prevention rule:** For multiclass CE, keep targets `(B,)`, not `(B,1)`.


In [ ]:
# Corrected runnable solution
fixed_logits = torch.randn(6, 4)
fixed_targets = torch.randint(0, 4, (6,), dtype=torch.long)
fixed_loss = nn.CrossEntropyLoss()(fixed_logits, fixed_targets)
assert fixed_loss.ndim == 0


### 🛠 Debug drill 4: Shape 4 — wrong argmax dimension

**Broken snippet**

```python
`predictions = logits.argmax(dim=0)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** It returns one winning example per class, shape `(C,)`, instead of one class per example.

**Prevention rule:** For `(B,C)` logits, reduce the class dimension: `dim=1`.


In [ ]:
# Corrected runnable solution
fixed_logits = torch.randn(6, 4)
fixed_predictions = fixed_logits.argmax(dim=1)
assert fixed_predictions.shape == (6,)


### 🛠 Debug drill 5: Shape 5 — unsafe bare squeeze

**Broken snippet**

```python
`logits = model(x).squeeze()` when `B == 1`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** The batch dimension can disappear, causing unstable shape contracts.

**Prevention rule:** Squeeze only the dimension you intend to remove.


In [ ]:
# Corrected runnable solution
fixed_binary_logits = torch.randn(1, 1)
fixed_binary_vector = fixed_binary_logits.squeeze(1)
assert fixed_binary_vector.shape == (1,)


### 🛠 Debug drill 6: Dtype 1 — float cross-entropy targets

**Broken snippet**

```python
`CrossEntropyLoss()(logits, targets.float())`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** A dtype error occurs because ordinary CE class indices must be long.

**Prevention rule:** Check `targets.dtype == torch.long` for multiclass CE.


In [ ]:
# Corrected runnable solution
fixed_logits = torch.randn(7, 3)
fixed_targets = torch.randint(0, 3, (7,), dtype=torch.long)
fixed_loss = nn.CrossEntropyLoss()(fixed_logits, fixed_targets)
assert torch.isfinite(fixed_loss)


### 🛠 Debug drill 7: Dtype 2 — integer features

**Broken snippet**

```python
`model(torch.tensor([[1, 2, 3]]))`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Linear layers normally have floating weights and reject integer inputs.

**Prevention rule:** Convert numeric model features to the floating dtype expected by the model.


In [ ]:
# Corrected runnable solution
fixed_features = torch.tensor([[1, 2, 3]], dtype=torch.float32)
fixed_output = nn.Linear(3, 2)(fixed_features)
assert fixed_output.dtype == torch.float32


### 🛠 Debug drill 8: Dtype 3 — binary target contract mismatch

**Broken snippet**

```python
`BCEWithLogitsLoss()(logits_B1, targets_B.long())`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Both dtype and shape are incompatible with `(B,1)` binary logits.

**Prevention rule:** BCE-with-logits targets should be floating and exactly match the logits shape.


In [ ]:
# Corrected runnable solution
fixed_binary_logits = torch.randn(7, 1)
fixed_binary_targets = torch.randint(0, 2, (7, 1)).float()
fixed_loss = nn.BCEWithLogitsLoss()(fixed_binary_logits, fixed_binary_targets)
assert fixed_loss.ndim == 0


### 🛠 Debug drill 9: Device 1 — model and features differ

**Broken snippet**

```python
`logits = model_on_device(features_on_cpu)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** On an accelerator this raises a device mismatch.

**Prevention rule:** Move the model and every input batch to compatible devices.


In [ ]:
# Corrected runnable solution
fixed_device_model = nn.Linear(3, 2).to(device)
fixed_device_features = torch.randn(4, 3).to(device)
fixed_device_output = fixed_device_model(fixed_device_features)
assert fixed_device_output.device == device


### 🛠 Debug drill 10: Device 2 — targets remain on CPU

**Broken snippet**

```python
`loss_fn(model(X.to(device)), y_cpu)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** On an accelerator, loss computation sees tensors on different devices.

**Prevention rule:** Move both features and targets inside each batch loop.


In [ ]:
# Corrected runnable solution
fixed_device_logits = torch.randn(4, 3, device=device)
fixed_device_targets = torch.randint(0, 3, (4,), device=device)
fixed_device_loss = nn.CrossEntropyLoss()(fixed_device_logits, fixed_device_targets)
assert fixed_device_loss.device == device


### 🛠 Debug drill 11: Device 3 — ignoring the return from `.to()`

**Broken snippet**

```python
`features.to(device); logits = model(features)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** The original tensor stays where it was because `.to()` is not an in-place move.

**Prevention rule:** Always assign the result: `x = x.to(device)`.


In [ ]:
# Corrected runnable solution
fixed_features = torch.randn(4, 3)
fixed_features = fixed_features.to(device)
assert fixed_features.device == device


### 🛠 Debug drill 12: Gradient 1 — missing zeroing (behavioral)

**Broken snippet**

```python
`loss.backward(); optimizer.step()` repeated without clearing
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Gradients silently accumulate and updates mix batches.

**Prevention rule:** Clear gradients before each normal independent training step.


In [ ]:
# Corrected runnable solution
fixed_grad_model = nn.Linear(3, 1)
fixed_grad_optimizer = torch.optim.SGD(fixed_grad_model.parameters(), lr=0.01)
fixed_grad_optimizer.zero_grad()
fixed_grad_loss = fixed_grad_model(torch.ones(2, 3)).pow(2).mean()
fixed_grad_loss.backward()
fixed_grad_optimizer.step()
assert all(p.grad is not None for p in fixed_grad_model.parameters())


### 🛠 Debug drill 13: Gradient 2 — training forward inside inference mode

**Broken snippet**

```python
`with torch.inference_mode(): logits = model(X); loss = loss_fn(logits, y); loss.backward()`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** The loss has no gradient path, so backward fails.

**Prevention rule:** Use inference mode for evaluation, never for the training forward pass.


In [ ]:
# Corrected runnable solution
fixed_grad_model = nn.Linear(3, 2)
fixed_grad_logits = fixed_grad_model(torch.randn(4, 3))
fixed_grad_loss = nn.CrossEntropyLoss()(fixed_grad_logits, torch.randint(0, 2, (4,)))
fixed_grad_loss.backward()
assert fixed_grad_model.weight.grad is not None


### 🛠 Debug drill 14: Gradient 3 — detached loss path

**Broken snippet**

```python
`loss = loss_fn(logits.detach(), targets)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** The model is disconnected, so backward cannot train its parameters.

**Prevention rule:** Detach only values used for logging or external inspection.


In [ ]:
# Corrected runnable solution
fixed_grad_model = nn.Linear(3, 2)
fixed_grad_logits = fixed_grad_model(torch.randn(4, 3))
fixed_grad_loss = nn.CrossEntropyLoss()(fixed_grad_logits, torch.randint(0, 2, (4,)))
fixed_grad_loss.backward()
assert fixed_grad_model.weight.grad is not None


### 🛠 Debug drill 15: Optimizer 1 — learning rate far too high (behavioral)

**Broken snippet**

```python
`AdamW(model.parameters(), lr=100.0)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Loss may explode, oscillate, or become non-finite.

**Prevention rule:** Start with a reasonable learning rate and inspect loss after one batch.


In [ ]:
# Corrected runnable solution
fixed_lr_model = nn.Linear(3, 2)
fixed_lr_optimizer = torch.optim.AdamW(fixed_lr_model.parameters(), lr=1e-3)
assert fixed_lr_optimizer.param_groups[0]["lr"] == 1e-3


### 🛠 Debug drill 16: Optimizer 2 — optimizer owns another model (behavioral)

**Broken snippet**

```python
`optimizer = AdamW(old_model.parameters()); loss = loss_fn(new_model(X), y)`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** The visible model gets gradients but its parameters never update.

**Prevention rule:** Construct the optimizer after the final model, from that exact model's parameters.


In [ ]:
# Corrected runnable solution
fixed_owned_model = nn.Linear(3, 2)
fixed_owned_optimizer = torch.optim.AdamW(fixed_owned_model.parameters(), lr=1e-3)
optimizer_parameter_ids = {id(p) for group in fixed_owned_optimizer.param_groups for p in group["params"]}
assert all(id(p) in optimizer_parameter_ids for p in fixed_owned_model.parameters())


### 🛠 Debug drill 17: Mode 1 — validation in training mode (behavioral)

**Broken snippet**

```python
`model.train(); with inference_mode(): validate()`
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Dropout remains random and stateful evaluation layers use training behavior.

**Prevention rule:** Validation requires both `eval()` and disabled gradient tracking.


In [ ]:
# Corrected runnable solution
fixed_mode_model = nn.Sequential(nn.Linear(3, 2), nn.Dropout(0.5))
fixed_mode_model.eval()
with torch.inference_mode():
    fixed_mode_output = fixed_mode_model(torch.ones(4, 3))
assert not fixed_mode_model.training


### 🛠 Debug drill 18: Mode 2 — training in evaluation mode (behavioral)

**Broken snippet**

```python
`model.eval();` followed by the training loop
```

**Before revealing:** inspect using the debugging order; name the expected error or behavioral symptom and the smallest fix.

**Symptom:** Code may run, but affected layers do not use training behavior.

**Prevention rule:** Call `model.train()` at the start of every training epoch.


In [ ]:
# Corrected runnable solution
fixed_mode_model = nn.Sequential(nn.Linear(3, 2), nn.Dropout(0.5))
fixed_mode_model.train()
assert fixed_mode_model.training


### Loss-not-decreasing checklist

1. Can the model overfit one tiny batch?
2. Are labels learnable and aligned with features?
3. Is the loss correct for output and target?
4. Are shapes and dtypes correct?
5. Do parameters have gradients?
6. Do parameters change after `step()`?
7. Is the learning rate reasonable?
8. Is the model in training mode?
9. Are hidden activations and the output layer appropriate?


## 12. Device Handling and Quick Save/Load

**🟡 Quick review · ⏱ 25 minutes**

Remember:

```python
model = model.to(device)
X = X.to(device)
y = y.to(device)
```

Parameters and input tensors must be on compatible devices. `.to(...)` returns the converted tensor; assign it. Move outputs to CPU when ordinary inspection needs it. This pattern still works when `device` is CPU.

### ✍️ Minimal `state_dict` save/load

Write the pattern in a scratch cell before running. The architecture must be instantiated first because a `state_dict` stores named tensors, not the Python class structure that consumes them.


In [ ]:
project_model.eval()
checkpoint_input = project_val_features[:8].to(device)
with torch.inference_mode():
    original_checkpoint_output = project_model(checkpoint_input)

with tempfile.TemporaryDirectory() as temporary_directory:
    checkpoint_path = Path(temporary_directory) / "tiny_model_state.pt"
    torch.save(project_model.state_dict(), checkpoint_path)

    restored_model = ProjectMLP()
    state = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )
    restored_model.load_state_dict(state)
    restored_model.to(device)
    restored_model.eval()
    with torch.inference_mode():
        restored_checkpoint_output = restored_model(checkpoint_input)
    assert torch.allclose(original_checkpoint_output, restored_checkpoint_output)

assert not checkpoint_path.exists()  # TemporaryDirectory cleaned the demonstration.


An optional fuller resume checkpoint may also store optimizer state and the epoch number. That is outside this sprint's required recall; the portable interview pattern is to save model weights and recreate the architecture before loading them.


## 13. Mock Coding Interview

**⏱ 60 minutes · no solutions until all four rounds are complete**

Fresh scenario: 900 rows, 9 floating features, and one binary outcome. You receive `mock_features` shaped `(900,9)` and labels derived from a learnable rule.

### Round A — Shape reasoning (10 minutes, 5 points)

1. What is one sample's feature shape? One target's shape?
2. What are batch shapes at batch size 30?
3. What output shape should the binary model produce?
4. What shape follows `(30,9) @ (9,12)`?
5. If logits are `(30,1)`, what does `squeeze(1)` produce and why avoid bare `squeeze()`?

### Round B — Model and loss (10 minutes, 6 points)

Define a `9 → 12 → 1` MLP. Justify the input size, output size, loss, target dtype/shape, and prediction rule.

### Round C — Training and evaluation (25 minutes, 12 points)

Write one complete training epoch and one evaluation function. Include device transfers and aggregate both loss and accuracy by example, including an incomplete last batch.

### Round D — Debugging and explanation (15 minutes, 7 points)

Diagnose, repair, name the symptom, and describe a test for each:

```python
# Bug 1
logits = model(features)
loss = nn.BCEWithLogitsLoss()(torch.sigmoid(logits), targets.long())

# Bug 2
model.eval()
for features, targets in val_loader:
    loss = loss_fn(model(features), targets)
    loss.backward()
    optimizer.step()
```

Stop here until the 60-minute attempt is complete.


### Mock reference solution — all rounds

**Round A:** `(9,)`, scalar target; `(30,9)` and `(30,1)`; output `(30,1)`; hidden `(30,12)`; `squeeze(1)` gives `(30,)`, while bare squeeze could erase batch size 1.

**Round B:** one raw logit per row, floating target matching `(B,1)`, BCE-with-logits, then sigmoid and threshold. The code below is a complete runnable reference for Rounds B and C.


In [ ]:
torch.manual_seed(SEED + 1)
mock_features = torch.randn(900, 9)
mock_rule = torch.randn(9, 1)
mock_targets = ((mock_features @ mock_rule + 0.10 * torch.randn(900, 1)) > 0).float()
mock_train_dataset = TensorDataset(mock_features[:720], mock_targets[:720])
mock_val_dataset = TensorDataset(mock_features[720:], mock_targets[720:])
mock_train_loader = DataLoader(
    mock_train_dataset, batch_size=30, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(SEED + 1),
)
mock_val_loader = DataLoader(mock_val_dataset, batch_size=30, shuffle=False, num_workers=0)


class MockBinaryMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(9, 12), nn.ReLU(), nn.Linear(12, 1))

    def forward(self, features):
        return self.net(features)


def train_binary_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for features, targets in loader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        count = features.size(0)
        total_loss += loss.item() * count
        total_correct += (predictions == targets).sum().item()
        total_examples += count
    return total_loss / total_examples, total_correct / total_examples


def evaluate_binary(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.inference_mode():
        for features, targets in loader:
            features, targets = features.to(device), targets.to(device)
            logits = model(features)
            loss = loss_fn(logits, targets)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            count = features.size(0)
            total_loss += loss.item() * count
            total_correct += (predictions == targets).sum().item()
            total_examples += count
    return total_loss / total_examples, total_correct / total_examples


mock_model = MockBinaryMLP().to(device)
mock_loss_fn = nn.BCEWithLogitsLoss()
mock_optimizer = torch.optim.AdamW(mock_model.parameters(), lr=5e-3)
for _ in range(5):
    mock_train_metrics = train_binary_epoch(
        mock_model, mock_train_loader, mock_loss_fn, mock_optimizer, device
    )
mock_val_metrics = evaluate_binary(mock_model, mock_val_loader, mock_loss_fn, device)
assert mock_val_metrics[0] > 0 and 0.0 <= mock_val_metrics[1] <= 1.0


**Round D reference:** Bug 1 passes probabilities rather than raw logits and integer targets rather than matching floats. Remove sigmoid from the loss path and use floating `(B,1)` targets; test a scalar finite loss and non-`None` parameter gradients. Bug 2 creates gradients and updates parameters during validation. Use `eval()` plus inference mode and remove backward/step; test by copying the state before validation and comparing every parameter afterward.

### 30-point rubric

| Area | Points | Full-credit evidence |
|---|---:|---|
| Shape reasoning | 5 | all batch, hidden, output, and squeeze shapes correct |
| Model/loss correctness | 6 | registered model, one logit, float matching targets, raw-logit loss |
| Training code | 7 | correct order, mode, devices, weighted metrics |
| Evaluation code | 5 | eval + inference, no updates, weighted metrics |
| Debugging | 4 | both root causes, symptoms, minimal repairs, tests |
| Communication/clean code | 3 | names assumptions and explains concise tradeoffs |

**26–30:** interview ready for core PyTorch tasks. **21–25:** nearly ready; repeat the weakest round. **15–20:** repeat the project and loop drills. **0–14:** return to Day 1 essentials.


## 14. Rapid-Fire Interview Questions

**⏱ 45 minutes · 🧠 answer each aloud before reading**

1. **What is a tensor?** A typed, multidimensional array with shape, dtype, device, and optional gradient tracking.
2. **What is a logit?** A raw model score before a probability conversion.
3. **What does `requires_grad` do?** It tells Autograd to track operations needed to differentiate with respect to that tensor.
4. **What happens when `loss.backward()` runs?** Autograd traverses the graph and accumulates derivatives into leaf parameter `.grad` fields.
5. **Why do gradients accumulate?** Additive accumulation supports use cases such as multiple loss contributions; ordinary independent steps must clear old gradients.
6. **Why call `optimizer.zero_grad()`?** To prevent the current step's gradients from being added to previous ones.
7. **What does `optimizer.step()` do?** It updates the optimizer's parameters using their current gradients and optimizer state.
8. **Epoch versus batch?** A batch is one subset used for a step; an epoch is one pass through the training dataset.
9. **Why mini-batches?** They bound memory use and provide frequent, somewhat noisy updates that work well in practice.
10. **Why subclass `nn.Module`?** It registers parameters/submodules and provides the standard model interface and mode/device behavior.
11. **Why call `super().__init__()`?** It initializes `nn.Module`'s internal registration machinery.
12. **What is `forward()`?** It defines how inputs are transformed into outputs.
13. **Why `model(x)` rather than `model.forward(x)`?** The module call path invokes `forward` while preserving hooks and framework bookkeeping.
14. **Why nonlinear activations?** Without them, stacked linear layers still represent one linear transformation.
15. **What is required by `CrossEntropyLoss`?** Raw `(B,C)` logits and typically `(B,)` long class indices.
16. **Why no softmax before cross entropy?** The loss operates on logits with a stable combined calculation; pre-softmax is redundant and can harm optimization.
17. **Why `BCEWithLogitsLoss`?** It combines the binary objective with logits handling more stably than separate sigmoid and BCE.
18. **Dataset versus DataLoader?** Dataset defines length and one-sample retrieval; DataLoader batches, shuffles, and iterates.
19. **Why shuffle training data?** It reduces order effects and changes batch composition across epochs.
20. **What does `model.train()` do?** It selects training behavior for affected layers; it does not itself perform training.
21. **What does `model.eval()` do?** It selects evaluation behavior for affected layers; it does not disable gradients.
22. **Does `eval()` disable gradients?** No; use no-grad or inference mode separately.
23. **`no_grad()` versus `inference_mode()`?** Both disable gradient recording; inference mode removes more Autograd overhead and is appropriate when tensors will not later participate in tracked computation.
24. **Why must model and data share a device?** Operations require participating tensors to live on compatible devices.
25. **`reshape` versus `view`?** `view` requires compatible strides and shares storage; `reshape` can return a view or make a copy when needed.
26. **`cat` versus `stack`?** `cat` grows an existing dimension; `stack` creates a new one.
27. **Why might loss not decrease?** Check data/label alignment, task contract, gradients, updates, learning rate, mode, and model capacity—starting with one batch.
28. **How verify parameters update?** Clone parameters before a step and compare afterward; also inspect gradients.
29. **How debug a shape mismatch?** Print shapes at each boundary and work forward from input to output/target contract.
30. **What is a `state_dict`?** A mapping from registered parameter/buffer names to tensors, used for portable weight saving and loading.

### Five scenario follow-ups

1. CE logits `(16,5)`, targets `(16,1)` long: reshape targets to `(16,)`; do not squeeze indiscriminately.
2. Binary logits `(16,1)`, targets `(16,)` float: make shapes match, for example targets `(16,1)`.
3. Validation differs between repeated calls: confirm `eval()` for stochastic/stateful layers, not only inference mode.
4. Gradients exist but weights do not change: verify `step()` and that the optimizer owns this model's parameters.
5. Loss is a little wrong when the last batch is smaller: weight batch means by batch size, then divide by total examples.


## 15. Final Memory Sheet

**10–15 minute last-minute review · highest-value syntax only**

### Device

```python
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
```

### Model

```python
class TabularMLP(nn.Module):
    def __init__(self, num_features, hidden_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        return self.net(x)
```

### Dataset and DataLoader

```python
class TabularDataset(Dataset):
    def __init__(self, features, targets):
        self.features, self.targets = features, targets
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.targets[index]

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0
)
```

### Loss choice

| Task | Output / target | Target dtype | Loss / prediction |
|---|---|---|---|
| Regression | `(B,1)` / `(B,1)` | float | MSE / raw |
| Binary | `(B,1)` / `(B,1)` | float | BCE-with-logits / sigmoid ≥ .5 |
| Multiclass | `(B,C)` / `(B,)` | long | CE / `argmax(1)` |
| Multilabel | `(B,C)` / `(B,C)` | float | BCE-with-logits / per-class threshold |

### Training

```python
model.train()
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
```

### Validation

```python
model.eval()
with torch.inference_mode():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        predictions = logits.argmax(dim=1)
```

### Prediction

```python
# Multiclass
predictions = logits.argmax(dim=1)

# Binary
probabilities = torch.sigmoid(logits)
predictions = (probabilities >= 0.5).long()
```

### Save/load

```python
torch.save(model.state_dict(), checkpoint_path)
state = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored_model.load_state_dict(state)
restored_model.to(device)
restored_model.eval()
```

### Debug order

`shape → dtype → device → gradients → mode → loss compatibility → parameter update`

### Ten rules not to forget

1. Cross entropy takes raw `(B,C)` logits.
2. Cross-entropy targets are `(B,)` and `torch.long`.
3. BCE-with-logits takes raw logits and floating-point targets.
4. PyTorch gradients accumulate by default.
5. Clear gradients before a normal independent training step.
6. Backward computes gradients; step updates parameters.
7. `eval()` does not disable gradients.
8. Validation uses both evaluation mode and disabled gradient tracking.
9. Model, features, and targets must be on compatible devices.
10. Dataset gives samples; DataLoader gives batches.


## 16. Readiness Scorecard

### I can do this without notes:

- [ ] predict common tensor-operation shapes
- [ ] explain broadcasting
- [ ] explain `cat` vs `stack`
- [ ] explain gradient accumulation
- [ ] build a tabular MLP with `nn.Module`
- [ ] determine the correct output layer size
- [ ] select regression, binary, and multiclass losses
- [ ] state the correct target shape and dtype
- [ ] write a training loop
- [ ] write a validation loop
- [ ] aggregate loss correctly
- [ ] calculate multiclass accuracy
- [ ] explain `train()` vs `eval()`
- [ ] explain `eval()` vs `inference_mode()`
- [ ] write a custom Dataset
- [ ] create train and validation DataLoaders
- [ ] write device-agnostic code
- [ ] debug shape, dtype, and device errors
- [ ] verify gradients and parameter updates
- [ ] save and restore a `state_dict`
- [ ] explain an end-to-end project in two minutes

> **Any unchecked Tier 1 skill should be practiced again before spending time on lower-priority topics.**

### Final 15-minute closed-book task

From an empty cell, write:

1. a three-class tabular MLP
2. `CrossEntropyLoss` and AdamW setup
3. a training loop
4. a validation loop
5. a custom Dataset

Then explain every expected shape and dtype aloud.
